In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv("../data/Loan_Default.csv")

# ทำความสะอาดเบื้องต้น (สรุปจาก Part 3)
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df = df.drop_duplicates()
df.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,75.135870,North,direct,1,39.0
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values -> {df[col].unique()[:5]}")
    print()

loan_limit: 2 unique values -> <StringArray>
['cf', 'ncf']
Length: 2, dtype: str

Gender: 4 unique values -> <StringArray>
['Sex Not Available', 'Male', 'Joint', 'Female']
Length: 4, dtype: str

approv_in_adv: 2 unique values -> <StringArray>
['nopre', 'pre']
Length: 2, dtype: str

loan_type: 3 unique values -> <StringArray>
['type1', 'type2', 'type3']
Length: 3, dtype: str

loan_purpose: 4 unique values -> <StringArray>
['p1', 'p4', 'p3', 'p2']
Length: 4, dtype: str

Credit_Worthiness: 2 unique values -> <StringArray>
['l1', 'l2']
Length: 2, dtype: str

open_credit: 2 unique values -> <StringArray>
['nopc', 'opc']
Length: 2, dtype: str

business_or_commercial: 2 unique values -> <StringArray>
['nob/c', 'b/c']
Length: 2, dtype: str

Neg_ammortization: 2 unique values -> <StringArray>
['not_neg', 'neg_amm']
Length: 2, dtype: str

interest_only: 2 unique values -> <StringArray>
['not_int', 'int_only']
Length: 2, dtype: str

lump_sum_payment: 2 unique values -> <StringArray>
['not_lpsm', 

In [ ]:
df_label = df.copy()
le = LabelEncoder()

# ตัวอย่าง: encode คอลัมน์ที่มีแค่ 2-3 ค่า
label_cols = ['Neg_ammortization', 'interest_only', 'lump_sum_payment']

for col in label_cols:
    df_label[col] = le.fit_transform(df_label[col].astype(str))
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

df_label[label_cols].head()

Neg_ammortization: {'neg_amm': np.int64(0), 'not_neg': np.int64(1)}
interest_only: {'int_only': np.int64(0), 'not_int': np.int64(1)}
lump_sum_payment: {'lpsm': np.int64(0), 'not_lpsm': np.int64(1)}


,Neg_ammortization,interest_only,lump_sum_payment
0,1,1,1
1,1,1,0
2,0,1,1
3,1,1,1
4,1,1,1


In [ ]:
df_onehot = df.copy()

onehot_cols = ['Gender', 'loan_purpose', 'Region', 'occupancy_type']

df_onehot = pd.get_dummies(df_onehot, columns=onehot_cols, drop_first=True)

# ดูคอลัมน์ใหม่ที่เกิดขึ้น
new_cols = [c for c in df_onehot.columns if any(c.startswith(o) for o in onehot_cols)]
df_onehot[new_cols].head()

,Gender_Joint,Gender_Male,Gender_Sex Not Available,loan_purpose_p2,loan_purpose_p3,loan_purpose_p4,Region_North-East,Region_central,Region_south,occupancy_type_pr,occupancy_type_sr
0,False,False,True,False,False,False,False,False,True,True,False
1,False,True,False,False,False,False,False,False,False,True,False
2,False,True,False,False,False,False,False,False,True,True,False
3,False,True,False,False,False,True,False,False,False,True,False
4,True,False,False,False,False,False,False,False,False,True,False


In [ ]:
df_final = df.copy()

# Label Encoding สำหรับ binary columns
for col in label_cols:
    df_final[col] = le.fit_transform(df_final[col].astype(str))

# One-Hot Encoding สำหรับ nominal columns
df_final = pd.get_dummies(df_final, columns=onehot_cols, drop_first=True)

print(f"Shape ก่อน encoding: {df.shape}")
print(f"Shape หลัง encoding: {df_final.shape}")
df_final.head()

Shape ก่อน encoding: (148670, 34)
Shape หลัง encoding: (148670, 41)


,ID,year,loan_limit,approv_in_adv,loan_type,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,rate_of_interest,...,Gender_Male,Gender_Sex Not Available,loan_purpose_p2,loan_purpose_p3,loan_purpose_p4,Region_North-East,Region_central,Region_south,occupancy_type_pr,occupancy_type_sr
0,24890,2019,cf,nopre,type1,l1,nopc,nob/c,116500,3.99,...,False,True,False,False,False,False,False,True,True,False
1,24891,2019,cf,nopre,type2,l1,nopc,b/c,206500,3.99,...,True,False,False,False,False,False,False,False,True,False
2,24892,2019,cf,pre,type1,l1,nopc,nob/c,406500,4.56,...,True,False,False,False,False,False,False,True,True,False
3,24893,2019,cf,nopre,type1,l1,nopc,nob/c,456500,4.25,...,True,False,False,False,True,False,False,False,True,False
4,24894,2019,cf,pre,type1,l1,nopc,nob/c,696500,4.00,...,False,False,False,False,False,False,False,False,True,False
